### **👩‍🏫 👩🏿‍🏫 What you will learn**

- How to control a web browser using Selenium to interact with JavaScript-heavy web pages.
- How to parse dynamically loaded content using `BeautifulSoup`.
- How to save scraped data into a structured format like a CSV file.

---

### **📃 Preparation**

- Install necessary Python libraries: `selenium`, `beautifulsoup4` (`bs4`), and `chromedriver-autoinstaller`.
- Choose a dynamic webpage for scraping. For this project, we will scrape dynamic product data from a demo e-commerce site, like: [inmotionhosting](https://www.inmotionhosting.com/).

### **Task**

- Initialize Selenium `WebDriver`
- Load the Web Page
- Identify the elements that contain hosting plan details.
- Extract necessary data such as plan names, features, and pricing.
- Store and Save the Data
- Close Selenium `WebDriver`


In [1]:
%pip install selenium beautifulsoup4 chromedriver-autoinstaller pandas

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [25]:
import chromedriver_autoinstaller
import pandas as pd
from bs4 import BeautifulSoup, Tag
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

In [5]:
HOME_URL = "https://www.inmotionhosting.com/"

### Initialize the Selenium WebDriver

The browser runs visibly (not headless) so we can watch it navigate from the home page to the shared hosting page. A fixed 1920x1080 window keeps the full desktop navigation bar on screen. In a narrow window it would collapse into a hamburger menu and the links we want to click would be hidden.


In [3]:
def init_driver(headless: bool = False) -> webdriver.Chrome:
    """
    Install a matching chromedriver and start a Chrome WebDriver session.

    Args:
        headless (bool, optional): Run Chrome without a visible window. Defaults to False.

    Returns:
        webdriver.Chrome: An open Chrome WebDriver session.
    """
    chromedriver_autoinstaller.install()
    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--window-size=1920,1080")

    return webdriver.Chrome(options=options)

In [4]:
driver = init_driver()

### Load the home page

We start where the instructions point, the InMotion Hosting home page. We wait for the `<body>` element so later steps don't run against a page that's still loading.


In [9]:
driver.get(HOME_URL)
wait = WebDriverWait(driver, 15)

In [10]:
wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))

<selenium.webdriver.remote.webelement.WebElement (session="f5cc6587f1cff37bf53553ce5ac6ecc7", element="f.B51C592383F49458940622C40ADEE94E.d.3CD01460A8843C00EDD541C86F931F89.e.3578")>

### Navigate to the Shared Hosting page

The home page footer lists every product inside a `<ul class="products-footer-menu">`. We look for the Shared Hosting link inside that list. The footer is far below the visible part of the page, and Chrome won't click an element that's off screen, so we scroll the link into view first. Then we wait until the URL changes, so the next step parses the new page and not the home page.


In [11]:
footer_link = wait.until(
    EC.presence_of_element_located(
        (By.CSS_SELECTOR, "ul.products-footer-menu a[href$='/shared-hosting']")
    )
)
driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", footer_link)
wait.until(EC.element_to_be_clickable(footer_link))
footer_link.click()

wait.until(EC.url_contains("/shared-hosting"))
print(driver.current_url)

https://www.inmotionhosting.com/shared-hosting


### Wait for the plan cards to render

The plan cards are rendered by JavaScript after the page loads, so the URL changing isn't enough. We wait until the `div.imh-rostrum-card` elements exist, then take `driver.page_source`, which is the HTML after JavaScript has run. A plain `requests.get` would only get the HTML from before that.


In [ ]:
cards = wait.until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.imh-rostrum-card"))
)
html = driver.page_source
print(f"Found {len(cards)} plan cards")

Found 4 plan cards


### Define the parsing functions

Each plan card holds four pricing blocks (`div.imh-switcher`), one per billing term. The term length is stored in the Select button's `data-term` attribute (36, 24, 12 or 1 months), and only the 12-month block carries the `active` class, which marks it as the price the page shows by default. Prices differ by term ($4.99/mo on 36 months, $4.79/mo on 12 months, $16.99 month-to-month), so scraping only the visible block would hide most of the pricing. We output one row per plan per term instead.

- **Prices:** each block has two `.rostrum-price` spans, the discounted price and the renewal price, which we tell apart by their parent div.
- **Total due:** the button's `data-presentation-price` attribute holds the upfront charge (57.48 = $4.79 × 12).
- **Features:** we join the `.imh-rostrum-detail` spans into one string. The tooltip explanations live in an `imh-tooltip-text` attribute, not in the text, so `get_text()` leaves them out.


In [27]:
def parse_price(text: str) -> float:
    """
    Convert a price string like "$4.99" into a float.

    Args:
        text (str): Price text as shown on the page.

    Returns:
        float: The numeric price.
    """
    return float(text.strip().lstrip("$").replace(",", ""))

In [31]:
def parse_card(card: Tag) -> list[dict]:
    """
    Extract one record per billing term from a single hosting plan card.

    Args:
        card (Tag): The div.imh-rostrum-card element for one plan.

    Returns:
        list[dict]: One dict per billing term with plan name, prices, savings and features.
    """
    plan_name = card.select_one(".imh-rostrum-card-title").get_text(strip=True)
    features = "; ".join(
        detail.get_text(" ", strip=True)
        for detail in card.select("ul.imh-rostrum-details-list .imh-rostrum-detail")
    )

    records = []

    for term_block in card.select("div.imh-switcher"):
        button = term_block.select_one("a.atc")
        savings = term_block.select_one(".imh-rostrum-save-ribbon .dnt").get_text(
            strip=True
        )
        records.append(
            {
                "plan": plan_name,
                "term_months": int(button["data-term"]),
                "monthly_price": parse_price(
                    term_block.select_one(
                        ".imh-rostrum-starting-at-price-discounted .rostrum-price"
                    ).get_text()
                ),
                "renewal_monthly_price": parse_price(
                    term_block.select_one(
                        ".imh-rostrum-starting-at-price-normal .rostrum-price"
                    ).get_text()
                ),
                "savings_pct": int(savings.rstrip("%")),
                "total_due": float(button["data-presentation-price"]),
                "default_term": "active" in term_block["class"],
                "features": features,
            }
        )

    return records

In [ ]:
def parse_plans(html: str) -> pd.DataFrame:
    """
    Parse every hosting plan card on the page into a tidy DataFrame.

    Args:
        html (str): Rendered page HTML from the Selenium driver.

    Returns:
        pd.DataFrame: One row per plan per billing term.
    """
    soup = BeautifulSoup(html, "html.parser")
    rows = [
        record
        for card in soup.select("div.imh-rostrum-card")
        for record in parse_card(card)
    ]

    return pd.DataFrame(rows)

### Extract the plan data


In [32]:
plans_df = parse_plans(html)
plans_df

,plan,term_months,monthly_price,renewal_monthly_price,savings_pct,total_due,default_term,features
0,Launch,36,4.99,13.99,64,179.64,False,2 Websites; 100GB NVMe Storage; Unmetered Band...
1,Launch,24,4.99,13.99,64,119.76,False,2 Websites; 100GB NVMe Storage; Unmetered Band...
2,Launch,12,4.79,14.99,68,57.48,True,2 Websites; 100GB NVMe Storage; Unmetered Band...
3,Launch,1,16.99,16.99,0,16.99,False,2 Websites; 100GB NVMe Storage; Unmetered Band...
4,Power,36,5.99,17.99,67,215.64,False,10 Websites; 200GB NVMe Storage; Unmetered Ban...
5,Power,24,5.99,17.99,67,143.76,False,10 Websites; 200GB NVMe Storage; Unmetered Ban...
6,Power,12,5.79,18.99,70,69.48,True,10 Websites; 200GB NVMe Storage; Unmetered Ban...
7,Power,1,17.99,20.99,14,17.99,False,10 Websites; 200GB NVMe Storage; Unmetered Ban...
8,Pro,36,10.99,24.99,56,395.64,False,40 Websites; 2 vCPU Cores; 4GB RAM; 300GB NVMe...
9,Pro,24,10.99,24.99,56,263.76,False,40 Websites; 2 vCPU Cores; 4GB RAM; 300GB NVMe...


### Save the data to CSV

`index=False` keeps the pandas row numbers out of the file, since they aren't part of the data.


In [33]:
plans_df.to_csv("inmotion_shared_hosting_plans.csv", index=False)

### Close the WebDriver

`quit()` ends the whole browser session and stops the chromedriver process. `close()` would only close the current tab and leave the chromedriver running in the background.


In [34]:
driver.quit()